In [ ]:
%load_ext autoreload
%autoreload 2

import os

import geopandas as gpd
import numpy as np

import src.data_filepaths as data_filepaths
import src.utils.loaders as loaders
import src.utils.vis.colors_utils as colors_utils
from src.utils.loaders import process_loaded_building_df
from src.utils.vis.map import CategoricalColorMap, MapExplorer

In [2]:
SRC_FPATH = r"./SG-Singapore_DATA/OSM_Overpass"

assert os.path.exists(SRC_FPATH) and os.path.isdir(SRC_FPATH), f"{SRC_FPATH} does not exist or isn't a directory"

In [3]:
b_gdf = gpd.read_parquet(data_filepaths.OVERTURE_BUILDING)
# b_gdf = building_gdf.to_crs("EPSG:3414")
bp_gdf = gpd.read_parquet(data_filepaths.OVERTURE_BUILDING_PART)

# Landuse

cuz it's relatively the easiest :)

All that matters *should* just be the `landuse=*` key...

In [4]:
landuse_fname = "SG_landuse_20260902.geojson"
landuse_fpath = os.path.join(SRC_FPATH, landuse_fname)

assert os.path.exists(landuse_fpath)

landuse_gdf = gpd.read_file(landuse_fpath)
print(len(landuse_gdf))

10133


In [5]:
# natively 262 columns, jeez
for c in landuse_gdf.columns:
    print(c)
landuse_gdf.head()

id
@id
abandoned
access
addr:city
addr:country
addr:housename
addr:housenumber
addr:place
addr:postcode
addr:province
addr:street
addr:suburb
aerodrome:type
aeroway
air_conditioning
alt_name
alt_name:nan
alt_name:nan-Latn-pehoeji
alt_name:nan-Latn-tailo
alt_name:ta
alt_name:zh
alt_name:zh-Hans
alt_name:zh-Hant
amenity
amenity:1992-2017
animal_shelter
animal_shelter:release
aquaculture
area
area:highway
barrier
bench
boundary
branch
brand
brand:wikidata
brand:wikipedia
bridge
bridge:structure
builder
building
building:colour
building:flats
building:levels
building:material
building:min_level
building:use
bunker_type
bus
capacity
cargo
cemetery
changing_table
check_date
chemical
city_served
closest_town
commercial
community_centre
compressed_air
condo
construction
contact:email
contact:facebook
contact:instagram
contact:phone
covered
crop
cuisine
denomination
depot
description
designation
developer
diet:halal
diet:vegetarian
disused
disused:leisure
disused:power
disused:school
dog
educat

,id,@id,abandoned,access,addr:city,addr:country,addr:housename,addr:housenumber,addr:place,addr:postcode,...,wikidata,wikidata:1992-2017,wikimedia_commons,wikipedia,@relations,wikipedia:1992-2017,wikipedia:en,year,year_of_construction,geometry
0,relation/2153897,relation/2153897,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((103.76231 1.41117, 103.76257 1..."
1,relation/2180209,relation/2180209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((103.81158 1.26283, 103.81162 1.26281..."
2,relation/2180210,relation/2180210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((103.81153 1.26291, 103.81162 1.26306..."
3,relation/2239678,relation/2239678,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((103.82738 1.3694, 103.82733 1...."
4,relation/2239768,relation/2239768,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((103.81308 1.37154, 103.813 1.3715, 1..."


In [6]:
print(landuse_gdf["landuse"].value_counts())

landuse
residential                3786
grass                      2434
construction                758
industrial                  659
forest                      555
commercial                  490
meadow                      331
religious                   248
retail                      161
military                    114
recreation_ground           106
allotments                   55
plant_nursery                55
garages                      54
farmland                     47
farmyard                     31
flowerbed                    29
aquaculture                  26
cemetery                     24
education                    23
greenfield                   20
brownfield                   18
railway                      11
healthcare                   11
village_green                11
greenhouse_horticulture      10
basin                        10
reservoir                     9
quarry                        8
disused                       6
institutional                 5


## Handling Multipolygon Confusions

or do I just not care about handling it at this super high level

In [7]:
print(landuse_gdf["geometry"].dtype)

geometry


## Landuse values to care about

* `construction`
* `port`
* `commercial` - for visualization
  
And then under each type I'd only care if the data is easy to deal with, heh

In [ ]:
landuse_gdf_clean = loaders.load_osm_landuse_1_geojson_to_gdf(landuse_fpath)
# print(landuse_gdf_clean["landuse"].value_counts())

In [18]:
def show_interesting_osm_landuses():
    # landuse_gdf_clean = loaders.load_osm_landuse_1_geojson_to_gdf(landuse_fpath)

    mapper = MapExplorer(
        building_gdf=process_loaded_building_df(b_gdf),
        building_part_gdf=process_loaded_building_df(bp_gdf),
        enable_debug_mode=True,
    )

    # cmap = colors_utils.CategoricalColorMap(
    #     cat_colors=[
    #         colors_utils.get_color_by_name("blue"),
    #         colors_utils.get_color_by_name("red"),
    #     ],
    #     cat_labels=[True, False],
    # )
    c_fn = colors_utils.get_color_by_name

    print(c_fn("lime"))

    landuse_gdf_clean["color"] = np.select(
        [
            landuse_gdf_clean["landuse"] == "residential",
            landuse_gdf_clean["landuse"] == "construction",
            (landuse_gdf_clean["landuse"] == "commercial") | (landuse_gdf_clean["landuse"] == "retail"),
            landuse_gdf_clean["landuse"] == "industrial",
            landuse_gdf_clean["landuse"] == "industrial_port",
        ],
        [
            c_fn("lime"),
            c_fn("mustard"),
            c_fn("ultramarine"),
            c_fn("grey"),
            c_fn("violet"),
        ],
        default=c_fn("white"),
    )

    return mapper.build_singleprop(
        column="class",
        # categorical_cmap=cmap,
        colored_landuse_gdf=landuse_gdf_clean,
        title="Landuse + Buildings Title",
    )


show_interesting_osm_landuses()

#00FF00


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich